In [ ]:
# Runpod GPU 실행 전 1회 설치 셀
# 이 셀을 먼저 실행한 뒤, 커널을 재시작하고 아래 셀부터 다시 순서대로 실행한다.
import sys
import subprocess

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-U',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu121',
    'torch==2.7.0',
    'torchvision==0.22.0',
    'torchaudio==2.7.0',
    'transformers>=4.53.3',
    'accelerate>=1.13.0',
    'huggingface_hub>=0.33.0',
    'safetensors',
    'sentencepiece',
    'protobuf',
    'bitsandbytes',
    'einops',
    'peft',
])

In [10]:
import json
import os
import re
import time
import urllib.request
import urllib.error
from collections import Counter

import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv(override=True)

print('라이브러리 로드 완료')

라이브러리 로드 완료


In [ ]:
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY','').strip()
OPENAI_MODEL = 'gpt-5-nano'
HF_MODEL = 'Qwen/Qwen3.6-27B'

print(OPENAI_API_KEY[:10] + '...'+OPENAI_API_KEY[-10:])

sk-proj-IF...uEOf8jXrMA


In [13]:
# 평가 데이터

sample_data = [
    {'id': 'D01', 'category': 'DEF', 'text': '이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.'},
    {'id': 'D02', 'category': 'DEF', 'text': '이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.'},
    {'id': 'D03', 'category': 'DEF', 'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.'},
    {'id': 'D04', 'category': 'DEF', 'text': '이 법은 대한민국 영역 안에서 이루어지는 정보 처리 행위에 적용한다.'},
    {'id': 'R01', 'category': 'RIGHT', 'text': '모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 아니한다.'},
    {'id': 'R02', 'category': 'RIGHT', 'text': '사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.'},
    {'id': 'R03', 'category': 'RIGHT', 'text': '근로자는 안전하고 건강한 근무 환경에서 일할 권리를 가진다.'},
    {'id': 'R04', 'category': 'RIGHT', 'text': '누구든지 정당한 사유 없이 타인의 통신비밀을 침해하여서는 아니 된다.'},
    {'id': 'P01', 'category': 'PROC', 'text': '이 법을 위반한 자는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처한다.'},
    {'id': 'P02', 'category': 'PROC', 'text': '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.'},
    {'id': 'P03', 'category': 'PROC', 'text': '장관은 위반 사실을 조사한 후 청문 절차를 거쳐 등록을 취소할 수 있다.'},
    {'id': 'P04', 'category': 'PROC', 'text': '불법행위로 인한 손해배상 청구는 민사소송법에서 정한 절차에 따른다.'},
    {'id': 'O01', 'category': 'ORG', 'text': '분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.'},
    {'id': 'O02', 'category': 'ORG', 'text': '위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.'},
    {'id': 'O03', 'category': 'ORG', 'text': '법원은 사법권을 행사하며 대법원, 고등법원 및 지방법원으로 구성된다.'},
    {'id': 'O04', 'category': 'ORG', 'text': '중앙행정기관의 장은 소관 사무를 관장하고 소속 공무원을 지휘한다.'},
    {'id': 'C01', 'category': 'CRIT', 'text': '후보자는 선거일 현재 25세 이상인 국민이어야 한다.'},
    {'id': 'C02', 'category': 'CRIT', 'text': '지원 자격은 해당 분야 경력 3년 이상 및 학사 학위 이상으로 한다.'},
    {'id': 'C03', 'category': 'CRIT', 'text': '안전관리 기준은 시설 면적, 이용 인원 및 위험도에 따라 대통령령으로 정한다.'},
    {'id': 'C04', 'category': 'CRIT', 'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.'},
    {'id': 'E01', 'category': 'ETC', 'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.'},
    {'id': 'E02', 'category': 'ETC', 'text': '이 법 시행 당시 종전의 규정에 따라 한 처분은 이 법에 따른 처분으로 본다.'},
    {'id': 'E03', 'category': 'ETC', 'text': '이 법의 시행에 필요한 사항은 대통령령으로 정한다.'},
    {'id': 'E04', 'category': 'ETC', 'text': '제3조의 개정규정은 이 법 시행 후 최초로 접수된 사건부터 적용한다.'},
]

label_desc = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항'
}

MAX_SAMPLES = 24

df_all = pd.DataFrame(sample_data)
df = df_all.head(MAX_SAMPLES).copy()
print(f'평가 데이터: {len(df)}건 / 전체 {len(df_all)}건')
df.head(5)

평가 데이터: 24건 / 전체 24건


,id,category,text
0,D01,DEF,이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.
1,D02,DEF,이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.
2,D03,DEF,"공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다."
3,D04,DEF,이 법은 대한민국 영역 안에서 이루어지는 정보 처리 행위에 적용한다.
4,R01,RIGHT,"모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 ..."


In [18]:
# 프롬프트 템플릿 정의(zero shot / one shot)
ALLOWED = list(label_desc.keys())

SYSTEM_PROMPT = (
    '너는 한국 법률 조항 분류기다, 반드시 다음 6개 코드 중 하나만 선택하라' + 
    ', '.join(ALLOWED) + '. '
    '응답은 반드시 JSON 한 줄로만 반환한다. 형식: {"label" : "DEF","reason":"..."}'
)

LABEL_GUIDE = '\n'.join([f'-{k}:{v}'  for k, v in label_desc.items()])

ONE_SHOT_EXAMPLE = {
    'text' : '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다',
    'output' : {'label':'PROC', 'reason' : '이의신청과 처리 절차를 규정한다'}
}

def build_user_prompt(text, mode='zero'):
    base = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하라.',
        '라벨 설명:',
        LABEL_GUIDE,
    ]
    if mode == 'one':
        base += [
            '',
            '예시 1개:',
            f'문장: {ONE_SHOT_EXAMPLE["text"]}',
            '정답(JSON): ' + json.dumps(ONE_SHOT_EXAMPLE['output'], ensure_ascii=False)
        ]
    base += ['', f'분류할 문장: {text}', 'JSON으로만 응답하라.']
    return '\n'.join(base)

print(build_user_prompt('위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.', mode='zero')[:300])


다음 법률 문장을 6개 코드 중 하나로 분류하라.
라벨 설명:
-DEF:정의/목적/적용범위 조항
-RIGHT:권리/의무/금지/책임 조항
-PROC:신청/심사/조사/불복/처벌 절차 조항
-ORG:기관/위원회/법원 등 조직의 설치/구성/권한 조항
-CRIT:자격/요건/기준/기간/수치 조건 조항
-ETC:시행일/경과조치/위임 등 기타 조항

분류할 문장: 위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.
JSON으로만 응답하라.


In [20]:
# openai 호출
from openai import OpenAI
client = OpenAI(api_key = OPENAI_API_KEY)

user_content = build_user_prompt('위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.', mode='zero')
messages = [
    {'role':'system','content':SYSTEM_PROMPT},
    {'role':'user','content':user_content}
]

response = client.responses.create(
    model=OPENAI_MODEL,
    input=messages
)

print(response.output_text)

{"label" : "ORG","reason":"위원회의 구성에 관한 조직 구성 조항(위원장 포함 15명 이내의 구성)"}


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = HF_MODEL

if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f'CUDA 사용 가능: {torch.cuda.get_device_name(0)}')
    print(f'로딩 dtype: {dtype}')
else:
    dtype = torch.float32
    print('CUDA를 찾지 못해 CPU로 로딩합니다.')

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map='auto',
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    attn_implementation='eager',
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

prompt = "Give me a short introduction to large language model."
# messages = [
#     {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
#     {"role": "user", "content": prompt}
# ]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{"label": "ORG", "reason": "이 문장은 기관 또는위원회에 의해 설정된 구성 조항입니다."}


In [22]:
# 허깅페이스모델로.. one shot / few shot을 만들어서 실습

In [ ]:
import json
from typing import Dict, List, Tuple

# Few-Shot 예시 데이터: 각 카테고리별 대표 예시 3개
# 과적합 유도를 위해 테스트 문장과 의도적으로 구분되는 패턴 사용
FEW_SHOT_EXAMPLES = [
    {
        'text': '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.',
        'output': {'label': 'PROC', 'reason': '신청/불복 절차를 규정하는 조항'}
    },
    {
        'text': '근로자는 안전하고 건강한 근무 환경에서 일할 권리를 가진다.',
        'output': {'label': 'RIGHT', 'reason': '근로자의 권리를 규정하는 조항'}
    },
    {
        'text': '분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.',
        'output': {'label': 'ORG', 'reason': '조직의 설치를 규정하는 조항'}
    }
]

def build_hf_prompt(text: str, mode: str = 'zero', examples: List[Dict] = None) -> str:
    """
    Hugging Face 모델용 프롬프트 생성 함수
    
    파라미터:
        text (str): 분류 대상 법률 문장
        mode (str): 프롬프팅 모드 ('zero'|'one'|'few')
        examples (List[Dict]): Few-Shot 예시 리스트
    
    반환값:
        str: 구성된 프롬프트 문자열
        
    설명:
        - base: 모든 모드에서 공통으로 포함할 라벨 설명과 지시사항
        - mode별로 예시를 추가하여 프롬프트 길이 증가
        - 최종적으로 '\n'으로 결합하여 가독성 확보
    """
    base = [
        "다음 법률 문장을 6개 코드 중 하나로 분류하세요.",
        "라벨 설명:",
        LABEL_GUIDE,
    ]
    
    if mode == 'one':
        base += [
            "",
            "[One-Shot 예시 1개]",
            f"문장: {ONE_SHOT_EXAMPLE['text']}",
            "정답(JSON): " + json.dumps(ONE_SHOT_EXAMPLE['output'], ensure_ascii=False)
        ]
    
    elif mode == 'few' and examples:
        base += ["", f"[Few-Shot 예시 {len(examples)}개]"]
        for i, ex in enumerate(examples, 1):
            base += [
                f"예시 {i}:",
                f"  문장: {ex['text']}",
                f"  정답: " + json.dumps(ex['output'], ensure_ascii=False)
            ]
    
    base += [
        "",
        f"분류할 문장: {text}",
        "응답 형식:",
        '{"label": "DEF|RIGHT|PROC|ORG|CRIT|ETC", "reason": "분류 근거"}'
    ]
    
    return "\n".join(base)

# 테스트 데이터: 실제 평가셋의 문장들 (Real Data)
test_sentences = [
    {
        'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
        'true_label': 'CRIT',
        'description': '자격/요건 규정'
    },
    {
        'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.',
        'true_label': 'ETC',
        'description': '시행일 규정'
    },
    {
        'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.',
        'true_label': 'DEF',
        'description': '정의 규정'
    }
]

results = []

print("=" * 80)
print("Hugging Face 모델 - One-Shot / Few-Shot 비교 실습")
print("=" * 80)
print(f"\n실습 설정:")
print(f"  - 모델: {model_name}")
print(f"  - 평가 문장: {len(test_sentences)}개")
print(f"  - Few-Shot 예시: {len(FEW_SHOT_EXAMPLES)}개")
print(f"  - 프롬프팅 모드: zero-shot, one-shot, few-shot (3가지)")
print()

for idx, test_data in enumerate(test_sentences, 1):
    test_text = test_data['text']
    true_label = test_data['true_label']
    
    print(f"\n[테스트 {idx}/{len(test_sentences)}] {test_data['description']}")
    print(f"문장: {test_text}")
    print(f"정답 레이블: {true_label}")
    print("-" * 80)
    
    mode_results = {}
    
    for mode in ['zero', 'one', 'few']:
        prompt = build_hf_prompt(test_text, mode=mode, examples=FEW_SHOT_EXAMPLES if mode == 'few' else None)
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
        
        text_input = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
        
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=256,
            temperature=0.3,
            top_p=0.9
        )
        
        generated_ids = [
            output_ids[len(input_ids):] 
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        mode_results[mode] = response
        
        mode_label_map = {'zero': 'Zero-Shot', 'one': 'One-Shot', 'few': 'Few-Shot'}
        print(f"\n{mode_label_map[mode]} 결과:")
        print(f"응답: {response[:180]}")
        
        try:
            json_start = response.find('{')
            json_end = response.rfind('}') + 1
            if json_start != -1 and json_end > json_start:
                predicted = json.loads(response[json_start:json_end])
                predicted_label = predicted.get('label', 'UNKNOWN')
                reason = predicted.get('reason', '')
                is_correct = 'O' if predicted_label == true_label else 'X'
                print(f"  예측 레이블: {predicted_label} ({is_correct})")
                print(f"  분류 근거: {reason}")
        except json.JSONDecodeError:
            print(f"  [JSON 파싱 실패]")
    
    results.append({
        'sentence': test_text,
        'true_label': true_label,
        'predictions': mode_results
    })

print("\n" + "=" * 80)
print("실습 완료: 예시 개수에 따른 성능 변화 확인")
print("=" * 80)
print("\n주요 관찰:")
print("  1. Zero-Shot: 사전학습된 지식만으로 분류 시도")
print("  2. One-Shot: 1개 예시로 분류 패턴 인지")
print("  3. Few-Shot: 여러 예시로 더 안정적인 분류")
print("\n평가 결과는 results 변수에 저장되었습니다.")

## 실습: Hugging Face 모델을 활용한 One-Shot/Few-Shot 프롬프팅

### 실습 목표
이 섹션에서는 법률 조항 분류 작업에 One-Shot과 Few-Shot 학습을 적용하여 프롬프트 엔지니어링의 성능 차이를 직접 확인한다.

### 핵심 개념
- **Zero-Shot**: 예시 없이 모델이 직접 분류 (사전학습 지식만 활용)
- **One-Shot**: 1개 예시를 제공하여 분류 패턴 학습
- **Few-Shot**: 여러 예시(보통 3-5개)를 제공하여 더 정확한 분류 유도

### 프롬프트 구조의 차이
각 모드에서 사용할 프롬프트는 라벨 설명(LABEL_GUIDE)과 예시(examples)의 포함 여부만 다르며, 나머지 지시사항과 분류 대상 문장은 동일하다. 이를 통해 예시의 영향을 순수하게 측정할 수 있다.

In [ ]:
# ========================================
# Hugging Face 파라미터 튜닝 실습
# ========================================
# 앞으로는 OpenAI 대신 Hugging Face 모델만 사용한다.
# 같은 프롬프트를 고정하고 generation 파라미터만 바꿔서 결과 차이를 비교한다.

HF_EXPERIMENTS = [
    {
        'name': 'deterministic',
        'do_sample': False,
        'temperature': None,
        'top_p': None,
        'max_new_tokens': 120,
        'description': '가장 안정적인 설정'
    },
    {
        'name': 'balanced',
        'do_sample': True,
        'temperature': 0.3,
        'top_p': 0.9,
        'max_new_tokens': 120,
        'description': '실무에서 자주 쓰는 중간 설정'
    },
    {
        'name': 'creative',
        'do_sample': True,
        'temperature': 0.8,
        'top_p': 0.95,
        'max_new_tokens': 120,
        'description': '변동성을 일부 허용하는 설정'
    }
]

TUNING_SAMPLES = [
    {
        'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
        'true_label': 'CRIT',
        'note': '수치 기준이 있는 요건 조항'
    },
    {
        'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.',
        'true_label': 'ETC',
        'note': '시행일을 정하는 조항'
    },
    {
        'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.',
        'true_label': 'DEF',
        'note': '개념의 뜻을 정의하는 조항'
    }
]


def extract_json_object(text):
    """모델 응답에서 JSON 객체만 안전하게 추출한다."""
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


def call_hf_classifier(text, do_sample, temperature, top_p, max_new_tokens):
    """같은 프롬프트에 대해 Hugging Face 생성 파라미터만 바꿔 호출한다.

    반환값:
        str: 모델의 원문 응답
    """
    prompt_text = build_hf_prompt(text, mode='zero')
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt_text}
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([chat_text], return_tensors='pt').to(model.device)

    generate_kwargs = {
        'max_new_tokens': max_new_tokens,
        'do_sample': do_sample,
        'pad_token_id': tokenizer.eos_token_id,
    }
    if temperature is not None:
        generate_kwargs['temperature'] = temperature
    if top_p is not None:
        generate_kwargs['top_p'] = top_p

    generated_ids = model.generate(**model_inputs, **generate_kwargs)
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


print('=' * 80)
print('Hugging Face 파라미터 튜닝 비교')
print('=' * 80)
print('고정 조건: 동일한 프롬프트, 동일한 문장, 동일한 출력 형식(JSON)')
print('변화 조건: temperature, top_p, max_new_tokens')

hf_results = []

for exp in HF_EXPERIMENTS:
    print('\n' + '-' * 80)
    print(f"실험: {exp['name']} | {exp['description']}")
    print(f"do_sample={exp['do_sample']}, temperature={exp['temperature']}, top_p={exp['top_p']}, max_new_tokens={exp['max_new_tokens']}")
    print('-' * 80)

    for sample in TUNING_SAMPLES:
        raw_response = call_hf_classifier(
            text=sample['text'],
            do_sample=exp['do_sample'],
            temperature=exp['temperature'],
            top_p=exp['top_p'],
            max_new_tokens=exp['max_new_tokens']
        )
        parsed = extract_json_object(raw_response)
        predicted_label = parsed.get('label') if parsed else 'PARSE_FAIL'
        reason = parsed.get('reason') if parsed else raw_response[:160]
        is_correct = predicted_label == sample['true_label']

        hf_results.append({
            'experiment': exp['name'],
            'do_sample': exp['do_sample'],
            'temperature': exp['temperature'],
            'top_p': exp['top_p'],
            'text': sample['text'],
            'true_label': sample['true_label'],
            'predicted_label': predicted_label,
            'correct': is_correct,
            'raw_response': raw_response,
        })

        print(f"\n문장: {sample['note']}")
        print(f"정답: {sample['true_label']} / 예측: {predicted_label} / 정답 여부: {'O' if is_correct else 'X'}")
        print(f"응답 일부: {raw_response[:180]}")
        print(f"이유: {reason}")

hf_result_df = pd.DataFrame(hf_results)
hf_summary = (
    hf_result_df
    .groupby(['experiment', 'do_sample', 'temperature', 'top_p'], as_index=False)['correct']
    .mean()
    .rename(columns={'correct': 'accuracy'})
    .sort_values('accuracy', ascending=False)
)

print('\n' + '=' * 80)
print('실험 요약')
print('=' * 80)
hf_summary

print('\n해석 기준')
print('- temperature가 낮을수록 출력이 더 안정적이고 반복성이 높다.')
print('- top_p가 낮을수록 상위 확률 토큰만 선택되어 보수적인 응답이 나온다.')
print('- max_new_tokens는 답변 길이를 제한하므로 JSON 형식을 안정적으로 유지하는 데 도움이 된다.')
print('\n다음 단계에서는 같은 설정을 유지한 채 few-shot 예시 수만 바꿔서 성능 변화를 비교하면 된다.')

## 다음 실습: Few-Shot 예시 개수에 따른 성능 비교

### 실습 목표
Hugging Face 모델에서 예시를 0개, 1개, 3개, 5개로 바꿔가며 법률 조항 분류 성능이 어떻게 달라지는지 확인한다.

### 왜 이 실습이 필요한가
앞 단계에서는 생성 파라미터의 영향을 확인했다. 이제는 프롬프트의 핵심 요소인 예시 개수 자체가 성능에 어떤 변화를 주는지 비교해야 한다. 이 실습은 Zero-Shot, One-Shot, Few-Shot의 차이를 더 분명하게 보여준다.

### 관찰 포인트
- 예시가 전혀 없을 때와 1개 있을 때의 차이
- 3개, 5개로 늘렸을 때의 정확도 변화
- 예시 수가 늘어날수록 항상 좋아지는지 여부
- 어떤 라벨에서 혼동이 자주 발생하는지

In [ ]:
# ========================================
# Few-Shot 예시 개수별 성능 비교
# ========================================
# 같은 Hugging Face 모델에서 예시 수만 바꿔가며 성능 변화를 확인한다.
# 핵심 질문은 '예시를 더 넣으면 항상 좋아지는가?' 이다.

FEW_SHOT_VARIANTS = {
    0: [],
    1: [FEW_SHOT_EXAMPLES[0]],
    3: FEW_SHOT_EXAMPLES[:3],
    5: FEW_SHOT_EXAMPLES + [
        {
            'text': '이 법 시행에 필요한 사항은 대통령령으로 정한다.',
            'output': {'label': 'ETC', 'reason': '시행에 필요한 사항을 위임하는 조항'}
        },
        {
            'text': '모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 아니한다.',
            'output': {'label': 'RIGHT', 'reason': '평등권과 차별 금지를 규정하는 조항'}
        }
    ]
}

EVAL_SAMPLES = [
    {
        'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
        'true_label': 'CRIT',
        'note': '수치 기준이 있는 요건 조항'
    },
    {
        'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.',
        'true_label': 'ETC',
        'note': '시행일을 정하는 조항'
    },
    {
        'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.',
        'true_label': 'DEF',
        'note': '개념의 뜻을 정의하는 조항'
    },
    {
        'text': '누구든지 정당한 사유 없이 타인의 통신비밀을 침해하여서는 아니 된다.',
        'true_label': 'RIGHT',
        'note': '금지와 보호의무를 함께 가진 조항'
    },
    {
        'text': '장관은 위반 사실을 조사한 후 청문 절차를 거쳐 등록을 취소할 수 있다.',
        'true_label': 'PROC',
        'note': '조사와 청문 절차가 포함된 조항'
    }
]


def build_hf_fewshot_prompt(text, examples):
    """예시 목록을 받아 Few-Shot 프롬프트를 만든다.

    반환값:
        str: 예시 개수가 반영된 최종 프롬프트
    """
    return build_hf_prompt(text, mode='few' if examples else 'zero', examples=examples)


def predict_with_hf(text, examples):
    """Hugging Face 모델로 단일 문장을 분류한다.

    반환값:
        dict: label, reason, raw_response를 포함한 결과
    """
    prompt = build_hf_fewshot_prompt(text, examples)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt}
    ]
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([chat_text], return_tensors='pt').to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    parsed = extract_json_object(raw_response)
    return {
        'raw_response': raw_response,
        'label': parsed.get('label') if parsed else 'PARSE_FAIL',
        'reason': parsed.get('reason') if parsed else raw_response[:160]
    }


print('=' * 90)
print('Few-Shot 예시 개수별 성능 비교')
print('=' * 90)
print('고정 조건: 모델, 평가 문장, 출력 형식, generation 설정')
print('변화 조건: 예시 개수(0, 1, 3, 5)')

fewshot_results = []

for example_count, examples in FEW_SHOT_VARIANTS.items():
    print('\n' + '-' * 90)
    print(f'예시 개수: {example_count}개')
    print('-' * 90)
    
    correct_count = 0
    for sample in EVAL_SAMPLES:
        result = predict_with_hf(sample['text'], examples)
        predicted_label = result['label']
        is_correct = predicted_label == sample['true_label']
        correct_count += int(is_correct)
        
        fewshot_results.append({
            'example_count': example_count,
            'text': sample['text'],
            'true_label': sample['true_label'],
            'predicted_label': predicted_label,
            'correct': is_correct,
            'raw_response': result['raw_response']
        })
        
        print(f"문장: {sample['note']}")
        print(f"정답: {sample['true_label']} / 예측: {predicted_label} / 정답 여부: {'O' if is_correct else 'X'}")
        print(f"이유: {result['reason']}")
        print('-' * 40)
    
    accuracy = correct_count / len(EVAL_SAMPLES)
    print(f'예시 {example_count}개 정확도: {accuracy:.2f}')

fewshot_df = pd.DataFrame(fewshot_results)
fewshot_summary = (
    fewshot_df
    .groupby('example_count', as_index=False)['correct']
    .mean()
    .rename(columns={'correct': 'accuracy'})
    .sort_values('example_count')
)

print('\n' + '=' * 90)
print('예시 개수별 요약')
print('=' * 90)
fewshot_summary

print('\n해석 포인트')
print('- 0개와 1개의 차이를 보면 one-shot의 최소 효과를 볼 수 있다.')
print('- 3개와 5개의 차이를 보면 few-shot 증가가 실제로 도움이 되는지 확인할 수 있다.')
print('- 특정 예시가 반복될수록 그 라벨로 편향되는지 확인할 수 있다.')
print('- 결과가 단순히 예시 개수만이 아니라 예시의 질에도 영향을 받는다는 점을 관찰할 수 있다.')

## 다음 실습: HF 성능을 올리는 핵심 전략 실습

### 실습 목표
예시를 더 넣는 대신, 헷갈리는 라벨 경계 규칙을 프롬프트에 명시하면 성능이 어떻게 달라지는지 비교한다.

### 왜 중요한가
분류 문제에서 성능 향상은 단순히 샘플 수를 늘리는 것보다, 모델이 혼동하는 기준을 선명하게 만드는 데서 더 크게 나온다. 이 실습에서는 오답 분석 관점의 규칙 강화가 효과적인지 확인한다.

### 관찰 포인트
- 기본 프롬프트와 규칙 강화 프롬프트의 차이
- `PROC`, `ETC`, `RIGHT`, `CRIT`처럼 경계가 애매한 라벨에서의 변화
- 출력 형식 고정이 안정성에 미치는 영향

In [ ]:
# ========================================
# HF 성능 향상을 위한 규칙 강화 전략 비교
# ========================================
# 핵심 전략: 예시 개수 확대보다, 헷갈리는 라벨 경계를 명시적으로 적어준다.
# 목표: baseline zero-shot과 rule-aware prompt의 분류 성능 차이를 확인한다.

RULES_FOR_CONFUSION = [
    'DEF는 정의, 목적, 적용범위처럼 개념을 설명하는 문장이다.',
    'RIGHT는 권리, 의무, 금지, 책임처럼 행위의 허용/금지를 다루는 문장이다.',
    'PROC는 신청, 심사, 조사, 불복, 처벌처럼 절차나 처리 흐름을 다루는 문장이다.',
    'ORG는 위원회, 법원, 기관, 조직의 설치, 구성, 권한을 다루는 문장이다.',
    'CRIT는 자격, 요건, 기준, 기간, 수치 조건처럼 충족해야 할 조건을 다루는 문장이다.',
    'ETC는 시행일, 경과조치, 위임, 부칙처럼 나머지 행정적 규정을 다루는 문장이다.',
    'CRIT와 ETC가 헷갈리면, 숫자나 요건이 있더라도 시행일과 경과조치는 ETC로 우선 판단한다.',
    'PROC와 ETC가 헷갈리면, 절차 자체를 직접 설명하면 PROC, 시행 시기나 부칙이면 ETC로 판단한다.'
]

RULE_AWARE_EVAL = [
    {
        'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
        'true_label': 'CRIT',
        'note': '수치 요건 문장'
    },
    {
        'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.',
        'true_label': 'ETC',
        'note': '시행일 문장'
    },
    {
        'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.',
        'true_label': 'DEF',
        'note': '정의 문장'
    },
    {
        'text': '누구든지 정당한 사유 없이 타인의 통신비밀을 침해하여서는 아니 된다.',
        'true_label': 'RIGHT',
        'note': '금지 문장'
    },
    {
        'text': '장관은 위반 사실을 조사한 후 청문 절차를 거쳐 등록을 취소할 수 있다.',
        'true_label': 'PROC',
        'note': '조사와 청문 절차 문장'
    }
]


def build_rule_aware_prompt(text):
    """혼동 규칙을 추가한 프롬프트를 생성한다.

    반환값:
        str: 라벨 설명 + 혼동 규칙 + 대상 문장을 포함한 최종 프롬프트
    """
    base = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하라.',
        '라벨 설명:',
        LABEL_GUIDE,
        '',
        '구분 규칙:',
    ]
    base.extend([f'- {rule}' for rule in RULES_FOR_CONFUSION])
    base.extend([
        '',
        f'분류할 문장: {text}',
        '반드시 다음 JSON 형식으로만 답하라.',
        '{"label": "DEF|RIGHT|PROC|ORG|CRIT|ETC", "reason": "짧은 근거"}'
    ])
    return '\n'.join(base)


def predict_hf_from_prompt(prompt_text):
    """주어진 프롬프트로 Hugging Face 모델을 호출한다.

    반환값:
        dict: label, reason, raw_response를 포함한 결과
    """
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt_text}
    ]
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([chat_text], return_tensors='pt').to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    parsed = extract_json_object(raw_response)
    return {
        'raw_response': raw_response,
        'label': parsed.get('label') if parsed else 'PARSE_FAIL',
        'reason': parsed.get('reason') if parsed else raw_response[:160]
    }


print('=' * 90)
print('HF 성능 향상 전략: 규칙 강화 비교')
print('=' * 90)
print('전략 1: 기본 zero-shot 프롬프트')
print('전략 2: 경계 규칙을 명시한 rule-aware 프롬프트')

strategy_rows = []

for sample in RULE_AWARE_EVAL:
    baseline_prompt = build_hf_prompt(sample['text'], mode='zero')
    baseline_result = predict_hf_from_prompt(baseline_prompt)
    baseline_correct = baseline_result['label'] == sample['true_label']

    rule_prompt = build_rule_aware_prompt(sample['text'])
    rule_result = predict_hf_from_prompt(rule_prompt)
    rule_correct = rule_result['label'] == sample['true_label']

    strategy_rows.append({
        'strategy': 'baseline',
        'text': sample['text'],
        'true_label': sample['true_label'],
        'predicted_label': baseline_result['label'],
        'correct': baseline_correct,
        'reason': baseline_result['reason']
    })
    strategy_rows.append({
        'strategy': 'rule_aware',
        'text': sample['text'],
        'true_label': sample['true_label'],
        'predicted_label': rule_result['label'],
        'correct': rule_correct,
        'reason': rule_result['reason']
    })

    print('\n' + '-' * 90)
    print(f"문장: {sample['note']}")
    print(f"정답 레이블: {sample['true_label']}")
    print(f"baseline -> {baseline_result['label']} / {'O' if baseline_correct else 'X'} / {baseline_result['reason']}")
    print(f"rule-aware -> {rule_result['label']} / {'O' if rule_correct else 'X'} / {rule_result['reason']}")

strategy_df = pd.DataFrame(strategy_rows)
strategy_summary = (
    strategy_df
    .groupby('strategy', as_index=False)['correct']
    .mean()
    .rename(columns={'correct': 'accuracy'})
    .sort_values('accuracy', ascending=False)
)

print('\n' + '=' * 90)
print('전략별 요약')
print('=' * 90)
strategy_summary

print('\n실전 해석')
print('- HF 분류 성능은 예시 수보다 규칙 명시와 경계 구분에서 더 크게 좋아질 수 있다.')
print('- 특히 ETC, PROC, CRIT처럼 의미 경계가 넓은 라벨은 규칙 추가 효과가 크다.')
print('- 다음 단계는 오답만 따로 모아 규칙을 계속 보정하는 방식이다.')

## 다음 실습: 출력 형식 단순화 전략

### 실습 목표
작은 Hugging Face 모델은 label과 reason을 동시에 생성하게 하면 흔들리기 쉽다. 이번 실습에서는 reason을 빼고 label만 출력하게 했을 때 성능이 좋아지는지 확인한다.

### 왜 중요한가
분류 실습에서는 정답이 필요한 것은 label 하나다. 설명을 덧붙이게 하면 모델의 부담이 커지므로, 출력 형식을 단순하게 만들면 정확도가 오를 수 있다.

### 관찰 포인트
- `label + reason` 출력과 `label only` 출력의 차이
- JSON 파싱 안정성
- 오답률이 높은 라벨에서 단순화 효과가 있는지

In [ ]:
# ========================================
# 출력 형식 단순화: label only 전략 비교
# ========================================
# reason 생성을 제거하고 label 하나만 맞추는지 본다.
# 작은 모델은 이 방식에서 더 안정적으로 동작하는 경우가 많다.

LABEL_ONLY_EVAL = RULE_AWARE_EVAL


def build_label_only_prompt(text):
    """label만 출력하도록 요구하는 프롬프트를 만든다.

    반환값:
        str: label only 출력 규칙이 포함된 최종 프롬프트
    """
    base = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하라.',
        '라벨 설명:',
        LABEL_GUIDE,
        '',
        '추가 규칙:',
        'reason은 쓰지 말고 label만 JSON으로 반환하라.',
        '반드시 다음 형식만 출력하라.',
        '{"label": "DEF|RIGHT|PROC|ORG|CRIT|ETC"}',
        '',
        f'분류할 문장: {text}'
    ]
    return '\n'.join(base)


def predict_label_only(text):
    """label only 프롬프트로 Hugging Face 모델을 호출한다.

    반환값:
        dict: label과 raw_response를 포함한 결과
    """
    prompt_text = build_label_only_prompt(text)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt_text}
    ]
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([chat_text], return_tensors='pt').to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    parsed = extract_json_object(raw_response)
    return {
        'raw_response': raw_response,
        'label': parsed.get('label') if parsed else 'PARSE_FAIL'
    }


print('=' * 90)
print('출력 형식 단순화 전략 비교')
print('=' * 90)
print('전략 1: baseline zero-shot (label + reason)')
print('전략 2: label only 출력')

label_only_rows = []

for sample in LABEL_ONLY_EVAL:
    baseline_prompt = build_hf_prompt(sample['text'], mode='zero')
    baseline_result = predict_hf_from_prompt(baseline_prompt)
    baseline_correct = baseline_result['label'] == sample['true_label']

    label_only_result = predict_label_only(sample['text'])
    label_only_correct = label_only_result['label'] == sample['true_label']

    label_only_rows.append({
        'strategy': 'baseline_reason',
        'text': sample['text'],
        'true_label': sample['true_label'],
        'predicted_label': baseline_result['label'],
        'correct': baseline_correct
    })
    label_only_rows.append({
        'strategy': 'label_only',
        'text': sample['text'],
        'true_label': sample['true_label'],
        'predicted_label': label_only_result['label'],
        'correct': label_only_correct
    })

    print('\n' + '-' * 90)
    print(f"문장: {sample['note']}")
    print(f"정답 레이블: {sample['true_label']}")
    print(f"baseline -> {baseline_result['label']} / {'O' if baseline_correct else 'X'}")
    print(f"label_only -> {label_only_result['label']} / {'O' if label_only_correct else 'X'}")

label_only_df = pd.DataFrame(label_only_rows)
label_only_summary = (
    label_only_df
    .groupby('strategy', as_index=False)['correct']
    .mean()
    .rename(columns={'correct': 'accuracy'})
    .sort_values('accuracy', ascending=False)
)

print('\n' + '=' * 90)
print('출력 형식별 요약')
print('=' * 90)
label_only_summary

print('\n실전 해석')
print('- reason을 빼면 생성 부담이 줄어들어 label 정확도가 올라갈 수 있다.')
print('- 작은 HF 모델은 자유 서술보다 단일 라벨 선택에서 더 안정적이다.')
print('- 그래도 label이 틀리면, 그때는 모델 크기나 데이터 질을 점검해야 한다.')